# 실습. 구리배선 부식진단 — S-parameter 시각화 & Linear Regression

본 노트북에서는 **구리배선(CU)의 S-parameter**로부터 **부식도(corrosion degree)**를 예측하는 회귀 문제를 다룹니다.

## 학습 목표
1. `original_data`의 S-parameter를 불러와 시각화한다.
2. PyTorch로 **Basic Linear Regression** 모델을 직접 구현·학습한다.
3. 학습 곡선과 Actual vs Predicted 결과로 모델 성능을 해석한다.

## 데이터 개요
| 항목 | 설명 |
|---|---|
| 입력 | 주파수별 S-parameter (주로 S11 dB, 201개 주파수 포인트) |
| 출력 | corrosion degree (연속값, 약 1.4 ~ 96.6) |
| 메타데이터 | `data_order(corrosion_degree).csv` — sample_number, date, corrosion |
| 원본 파일 | `original_data/21{date}/CU_{sample}_{date}.csv` |
| 데이터 준비 | **아래 [0] 셀이 자동으로 내려받습니다** (업로드·경로입력 불필요) |

## 실행 방법 (수강생)
1. 상단 메뉴에서 **[드라이브에 사본 저장]** 을 눌러 본인 계정으로 복사하세요.
2. 위에서부터 셀을 **순서대로** 실행하면 됩니다.
3. Google Drive 마운트, 파일 업로드, 경로 수정은 **필요 없습니다.**
4. 선형회귀라 **CPU만으로 충분**합니다. (GPU를 써도 무방합니다)

## 0. 실습 데이터 자동 다운로드 & 환경 준비

공유된 `Cu_corrosion_dataset.zip` 을 내려받아 압축을 풀고, 그래프용 한글 폰트까지 함께 준비합니다.
셀을 여러 번 실행해도 이미 받아둔 데이터는 다시 내려받지 않습니다.

> Colab에는 PyTorch·pandas·scikit-learn이 기본 설치되어 있어 별도 설치가 필요 없습니다.

In [ ]:
# ============================================================
# [0] 실습 데이터 자동 다운로드 & 한글 폰트 설정
#  - Google Drive 공유 zip 을 내려받아 압축을 풉니다.
#  - Drive 마운트나 경로 입력이 필요 없습니다.
#  - 이미 준비되어 있으면 건너뜁니다.
# ============================================================
import sys, subprocess, zipfile, shutil
from pathlib import Path

ZIP_FILE_ID = "1x9nPoeAxmJ0I5ntfsHSmgEgE6DoZSdYz"   # Cu_corrosion_dataset.zip
IN_COLAB    = Path("/content").is_dir()

BASE        = Path("/content") if IN_COLAB else Path(".")
ZIP_PATH    = BASE / "Cu_corrosion_dataset.zip"
EXTRACT_DIR = BASE / "corrosion_data"


def find_data_dir(base: Path, max_depth: int = 3):
    """original_data 폴더와 data_order*.csv 를 함께 가진 디렉터리를 찾는다."""
    if not base.exists():
        return None
    queue = [(base, 0)]
    while queue:
        cur, depth = queue.pop(0)
        if (cur / "original_data").is_dir() and any(cur.glob("data_order*.csv")):
            return cur
        if depth < max_depth:
            for child in sorted(p for p in cur.iterdir() if p.is_dir()):
                if child.name in ("__MACOSX", ".ipynb_checkpoints"):
                    continue
                queue.append((child, depth + 1))
    return None


DATA_DIR = find_data_dir(EXTRACT_DIR)

if DATA_DIR is None:
    try:
        import gdown                     # Colab에는 기본 설치되어 있습니다
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"])
        import gdown

    if not ZIP_PATH.exists():
        print("데이터를 내려받는 중입니다. 잠시만 기다려 주세요...\n")
        gdown.download(id=ZIP_FILE_ID, output=str(ZIP_PATH), quiet=False)

    print("\n압축을 푸는 중입니다...")
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_DIR)

    shutil.rmtree(EXTRACT_DIR / "__MACOSX", ignore_errors=True)
    DATA_DIR = find_data_dir(EXTRACT_DIR)

if DATA_DIR is None:
    raise RuntimeError(
        "데이터 준비에 실패했습니다.\n"
        " 1) 공유 링크가 '링크가 있는 모든 사용자(뷰어)'로 설정되어 있는지 확인하세요.\n"
        " 2) 동시 접속이 많으면 잠시 후 다시 실행하면 되는 경우가 많습니다.\n"
        " 3) 그래도 안 되면 아래 줄을 실행해 처음부터 다시 받으세요.\n"
        "    !rm -rf ./corrosion_data ./Cu_corrosion_dataset.zip "
        "/content/corrosion_data /content/Cu_corrosion_dataset.zip"
    )

print("✅ 데이터 준비 완료")
print("DATA_DIR :", DATA_DIR)


# ---------- 그래프 한글 폰트 ----------
# Colab 기본 환경에는 한글 폰트가 없어 그래프 제목이 □□□ 로 깨집니다.
def setup_korean_font():
    import matplotlib
    import matplotlib.font_manager as fm

    nanum = Path("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    if IN_COLAB and not nanum.exists():
        print("한글 폰트를 설치하는 중입니다... (약 10~20초)")
        subprocess.run("apt-get -qq -y install fonts-nanum",
                       shell=True, capture_output=True)

    if nanum.exists():
        fm.fontManager.addfont(str(nanum))
        matplotlib.rcParams["font.family"] = "NanumGothic"
    else:
        # 로컬(Windows/macOS) 환경 대비 후보 폰트
        installed = {f.name for f in fm.fontManager.ttflist}
        for cand in ("NanumGothic", "Malgun Gothic", "AppleGothic", "NanumBarunGothic"):
            if cand in installed:
                matplotlib.rcParams["font.family"] = cand
                break
        else:
            print("[알림] 한글 폰트를 찾지 못했습니다. 그래프의 한글이 깨질 수 있습니다.")

    matplotlib.rcParams["axes.unicode_minus"] = False
    return matplotlib.rcParams["font.family"]


print("그래프 폰트 :", setup_korean_font())

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

# 한글 폰트는 위 [0] 셀에서 환경에 맞게 이미 설정되었습니다.
plt.rcParams['axes.unicode_minus'] = False

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__}')
print(f'Device : {device}')

## 1. 데이터 경로 및 메타데이터 확인

데이터 경로(`DATA_DIR`)는 위 [0] 셀에서 자동으로 잡혔습니다. **수정할 부분은 없습니다.**

`data_order(corrosion_degree).csv`는 각 시편의 **샘플 번호·측정일·부식도**를 담고 있습니다.
원본 S-parameter 파일 이름은 `CU_{sample_number}_{date}.csv` 형식입니다.

In [ ]:
# ============================================================
# [경로 설정] 위 [0] 셀에서 자동으로 잡힌 DATA_DIR 을 그대로 사용합니다.
# (Google Drive 마운트도, 경로 입력도 필요하지 않습니다)
# ============================================================
try:
    DATA_DIR
except NameError:
    raise RuntimeError("먼저 맨 위의 [0] 데이터 자동 다운로드 셀을 실행하세요.")

ORIGINAL_DIR = DATA_DIR / 'original_data'
_metas = sorted(DATA_DIR.glob('data_order*.csv'))

assert ORIGINAL_DIR.exists(), f'original_data 폴더가 없습니다: {ORIGINAL_DIR}'
assert _metas, f'data_order*.csv 메타데이터를 찾을 수 없습니다: {DATA_DIR}'
META_PATH = _metas[0]

print(f'DATA_DIR  = {DATA_DIR}')
print(f'META_PATH = {META_PATH.name}')

# date는 '0524'처럼 앞자리 0이 중요하므로 반드시 문자열로 읽습니다.
meta = pd.read_csv(META_PATH, dtype={'sample_number': int, 'date': str})
print(f'샘플 수: {len(meta)}')
print(meta.head(10))
print('\n부식도 통계:')
print(meta['corrosion'].describe())

## 2. Original Data 로드

각 CSV는 주파수(GHz)에 따른 S11/S12/S21/S22 (dB)와 Phase 값을 포함합니다.  
본 실습에서는 **S11 (dB)** 곡선을 주요 feature로 사용합니다. (필요하면 다른 채널도 추가 가능)

In [ ]:
FEATURE_COLUMNS = ['S11 (dB)']  # 확장 예: ['S11 (dB)', 'S21 (dB)', 'S12 (dB)', 'S22 (dB)']


def resolve_sample_path(sample_number: int, date: str) -> Path:
    """original_data/21{date}/CU_{sample}_{date}.csv 경로를 반환."""
    date = str(date).strip().zfill(4)  # '524' -> '0524'
    folder = ORIGINAL_DIR / f'21{date}'
    path = folder / f'CU_{int(sample_number)}_{date}.csv'
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def load_original_dataset(meta_df: pd.DataFrame, feature_columns=None):
    """메타데이터에 따라 original S-parameter와 부식도를 모두 로드.

    Returns
    -------
    frequency : (L,) ndarray
    X : (N, C, L) ndarray  — C=채널 수, L=주파수 포인트(보통 201)
    y : (N,) ndarray       — corrosion degree
    info : DataFrame       — sample_number, date, corrosion, path
    """
    feature_columns = feature_columns or FEATURE_COLUMNS
    X_list, y_list, rows = [], [], []
    frequency = None

    for _, row in meta_df.iterrows():
        path = resolve_sample_path(int(row['sample_number']), str(row['date']))
        df = pd.read_csv(path)

        if frequency is None:
            frequency = df['Frequency (GHz)'].to_numpy(dtype=np.float32)

        feat = df[feature_columns].to_numpy(dtype=np.float32).T  # (C, L)
        X_list.append(feat)
        y_list.append(float(row['corrosion']))
        rows.append({
            'sample_number': int(row['sample_number']),
            'date': str(row['date']),
            'corrosion': float(row['corrosion']),
            'path': str(path),
        })

    X = np.stack(X_list, axis=0)
    y = np.asarray(y_list, dtype=np.float32)
    info = pd.DataFrame(rows)
    return frequency, X, y, info


frequency, X, y, info = load_original_dataset(meta)
print(f'frequency shape : {frequency.shape}')
print(f'X shape         : {X.shape}  # (N, channels, freq_points)')
print(f'y shape         : {y.shape}')
print(f'부식도 범위     : {y.min():.3f} ~ {y.max():.3f}')

## 3. Original Data 시각화

부식도가 달라질 때 S-parameter 곡선이 어떻게 변하는지 관찰합니다.

### 3-1. 부식도 분포

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(y, bins=25, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Corrosion degree')
axes[0].set_ylabel('Count')
axes[0].set_title('부식도 히스토그램')

date_means = info.groupby('date')['corrosion'].mean().sort_index()
axes[1].bar(date_means.index.astype(str), date_means.values, color='darkorange')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Mean corrosion')
axes[1].set_title('측정일별 평균 부식도')
axes[1].tick_params(axis='x', rotation=45)

fig.tight_layout()
plt.show()

### 3-2. 부식도에 따른 S11 곡선

부식도가 낮은/중간/높은 시편을 골라 S11(dB) vs Frequency를 비교합니다.

In [ ]:
def pick_by_quantile(values, q):
    """부식도 분위수에 가장 가까운 샘플 인덱스."""
    target = np.quantile(values, q)
    return int(np.argmin(np.abs(values - target)))


idx_low = pick_by_quantile(y, 0.1)
idx_mid = pick_by_quantile(y, 0.5)
idx_high = pick_by_quantile(y, 0.9)

fig, ax = plt.subplots(figsize=(10, 5))
for idx, label in [
    (idx_low, f'Low  (corr={y[idx_low]:.2f})'),
    (idx_mid, f'Mid  (corr={y[idx_mid]:.2f})'),
    (idx_high, f'High (corr={y[idx_high]:.2f})'),
]:
    ax.plot(frequency, X[idx, 0], label=label, linewidth=2)

ax.set_xlabel('Frequency (GHz)')
ax.set_ylabel('S11 (dB)')
ax.set_title('부식도 수준별 S11 곡선 비교')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

print('선택된 샘플:')
print(info.loc[[idx_low, idx_mid, idx_high], ['sample_number', 'date', 'corrosion']])

### 3-3. 전체 샘플 S11 오버레이 (부식도 색상 맵)

모든 곡선을 겹쳐 그리고, 색으로 부식도를 표현합니다.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
norm = plt.Normalize(vmin=y.min(), vmax=y.max())
cmap = plt.cm.viridis

for i in range(len(X)):
    ax.plot(frequency, X[i, 0], color=cmap(norm(y[i])), alpha=0.35, linewidth=1)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('Corrosion degree')

ax.set_xlabel('Frequency (GHz)')
ax.set_ylabel('S11 (dB)')
ax.set_title('전체 샘플 S11 오버레이 (색 = 부식도)')
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()

### (선택)3-4. S11 히트맵 (샘플 × 주파수)

행을 부식도 오름차순으로 정렬하면, 부식도에 따른 스펙트럼 패턴 변화를 한눈에 볼 수 있습니다.

In [ ]:
order = np.argsort(y)
X_sorted = X[order, 0, :]  # (N, L)
y_sorted = y[order]

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(
    X_sorted,
    aspect='auto',
    origin='lower',
    extent=[frequency.min(), frequency.max(), 0, len(y_sorted)],
    cmap='magma',
)
ax.set_xlabel('Frequency (GHz)')
ax.set_ylabel('Samples (sorted by corrosion ↑)')
ax.set_title('S11 (dB) 히트맵 — 부식도 오름차순')
cbar = fig.colorbar(im, ax=ax)
cbar.set_label('S11 (dB)')

# 오른쪽 보조축: 부식도 눈금
ax2 = ax.twinx()
ax2.set_ylim(ax.get_ylim())
tick_pos = np.linspace(0, len(y_sorted) - 1, 5).astype(int)
ax2.set_yticks(tick_pos)
ax2.set_yticklabels([f'{y_sorted[i]:.1f}' for i in tick_pos])
ax2.set_ylabel('Corrosion degree')

fig.tight_layout()
plt.show()

## 4. Feature 준비 (Linear Regression용)

선형 회귀는 입력을 **1차원 벡터**로 받습니다.  
각 샘플의 S-parameter 곡선 `(C, L)`을 flatten하여 `(C*L,)` feature로 만듭니다.

예: S11만 사용 → feature 차원 = 201

In [ ]:
# (N, C, L) -> (N, C*L)
X_flat = X.reshape(X.shape[0], -1).astype(np.float32)
print(f'X_flat shape: {X_flat.shape}')

# Train / Test 분할 (80 / 20)
X_train, X_test, y_train, y_test = train_test_split(
    X_flat, y, test_size=0.2, random_state=SEED
)
print(f'train: {X_train.shape[0]}, test: {X_test.shape[0]}')

# 표준화 (train 통계만 사용! — 데이터 누수 방지)
feat_mean = X_train.mean(axis=0, keepdims=True)
feat_std = X_train.std(axis=0, keepdims=True) + 1e-8
label_mean = float(y_train.mean())
label_std = float(y_train.std() + 1e-8)

X_train_n = (X_train - feat_mean) / feat_std
X_test_n = (X_test - feat_mean) / feat_std
y_train_n = (y_train - label_mean) / label_std
y_test_n = (y_test - label_mean) / label_std

print(f'feature mean/std shape: {feat_mean.shape}')
print(f'label mean={label_mean:.3f}, std={label_std:.3f}')

## 5. PyTorch Basic Linear Regression

가장 단순한 회귀 모델입니다.

$$
\hat{y} = \mathbf{w}^\top \mathbf{x} + b
$$

- `nn.Linear(in_features, 1)` 한 층만 사용
- 손실함수: MSE Loss
- 최적화: Adam

> Tip: 레이블도 표준화해서 학습하면 수렴이 안정적입니다. 평가 시에는 원래 스케일로 되돌립니다.

In [ ]:
class LinearRegressor(nn.Module):
    """Basic Linear Regression: y = Wx + b"""

    def __init__(self, in_features: int):
        super().__init__()
        self.linear = nn.Linear(in_features, 1)

    def forward(self, x):
        # x: (B, F) -> (B,)
        return self.linear(x).squeeze(-1)


in_features = X_train_n.shape[1]
model = LinearRegressor(in_features).to(device)
print(model)
print(f'학습 파라미터 수: {sum(p.numel() for p in model.parameters())}')

In [ ]:
BATCH_SIZE = 32
EPOCHS = 200
LR = 1e-2

train_ds = TensorDataset(
    torch.from_numpy(X_train_n),
    torch.from_numpy(y_train_n),
)
test_ds = TensorDataset(
    torch.from_numpy(X_test_n),
    torch.from_numpy(y_test_n),
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

### 5-1. 학습 루프

매 epoch마다
1. forward → loss 계산
2. backward → gradient 계산
3. optimizer.step() → 가중치 갱신

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    """optimizer가 있으면 train, 없으면 eval."""
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = criterion(pred, yb)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * xb.size(0)

    return total_loss / len(loader.dataset)


history = {'train_loss': [], 'test_loss': []}

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, criterion, optimizer)
    test_loss = run_epoch(model, test_loader, criterion, optimizer=None)
    history['train_loss'].append(train_loss)
    history['test_loss'].append(test_loss)

    if epoch % 20 == 0 or epoch == 1:
        print(f'[{epoch:3d}/{EPOCHS}] train_mse={train_loss:.4f}  test_mse={test_loss:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history['train_loss'], label='Train MSE')
ax.plot(history['test_loss'], label='Test MSE')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE (normalized label)')
ax.set_title('학습 곡선')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 6. 평가 (원래 부식도 스케일)

표준화된 예측값을 원래 corrosion degree로 복원한 뒤 MAE / RMSE / R²를 계산합니다.

In [ ]:
@torch.no_grad()
def predict_numpy(model, X_np):
    model.eval()
    xb = torch.from_numpy(X_np).to(device)
    pred_n = model(xb).cpu().numpy()
    return pred_n * label_std + label_mean


y_train_pred = predict_numpy(model, X_train_n)
y_test_pred = predict_numpy(model, X_test_n)


def report_metrics(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    print(f'[{name}] MAE={mae:.3f}  RMSE={rmse:.3f}  R²={r2:.3f}')
    return mae, rmse, r2


_ = report_metrics('Train', y_train, y_train_pred)
_ = report_metrics('Test ', y_test, y_test_pred)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, yt, yp, title in [
    (axes[0], y_train, y_train_pred, 'Train: Actual vs Predicted'),
    (axes[1], y_test, y_test_pred, 'Test: Actual vs Predicted'),
]:
    ax.scatter(yt, yp, s=28, alpha=0.7, edgecolors='k', linewidths=0.3)
    lo = min(yt.min(), yp.min())
    hi = max(yt.max(), yp.max())
    ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1.5, label='y = x')
    ax.set_xlabel('Actual corrosion')
    ax.set_ylabel('Predicted corrosion')
    ax.set_title(title)
    ax.set_aspect('equal', adjustable='box')
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

### 6-1. 학습된 가중치 해석 (보너스)

선형 모델의 가중치 `|w|`가 큰 주파수 대역은 부식도 예측에 더 크게 기여합니다.

In [ ]:
weights = model.linear.weight.detach().cpu().numpy().reshape(-1)
n_freq = len(frequency)
n_channels = len(FEATURE_COLUMNS)

fig, axes = plt.subplots(n_channels, 1, figsize=(10, 3.2 * n_channels), sharex=True)
if n_channels == 1:
    axes = [axes]

for c, (ax, col) in enumerate(zip(axes, FEATURE_COLUMNS)):
    w_c = weights[c * n_freq:(c + 1) * n_freq]
    ax.plot(frequency, w_c, color='crimson')
    ax.axhline(0, color='gray', linewidth=1)
    ax.set_ylabel('Weight')
    ax.set_title(f'학습된 가중치 — {col}')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Frequency (GHz)')
fig.tight_layout()
plt.show()

## 7. 정리 & 다음 단계

이번 실습에서 한 일:
1. **Original S-parameter**를 부식도와 함께 시각화
2. S-parameter를 flatten한 feature로 **PyTorch Linear Regression** 학습
3. MAE / RMSE / R²와 Actual–Predicted 산점도로 성능 확인

### 스스로 해보기 (선택)
- `FEATURE_COLUMNS`에 `S21 (dB)` 등을 추가해 성능 변화를 비교해 보세요.
- `EPOCHS`, `LR`, `BATCH_SIZE`를 바꿔 학습 곡선이 어떻게 달라지는지 관찰해 보세요.
- Linear 대신 `nn.Sequential(nn.Linear(...), nn.ReLU(), nn.Linear(...))` MLP를 만들어 보세요.
- 고급: `data_and_code_sample/corrosion/code/`의 1D-CNN / ResNeXt 회귀와 성능을 비교해 보세요.

### 주의
- 본 노트북의 train/test 분할은 **샘플 단위 랜덤 분할**입니다.  
  같은 specimen이 train/test에 섞일 수 있어, 실제 일반화 성능은 다소 낙관적으로 보일 수 있습니다.
- 더 엄격한 평가는 specimen(시편 번호) 단위로 나누는 방식을 사용합니다. (`dataset.py`의 fold split 참고)